# ONT Data Processing

**Tier 3 — Applied Bioinformatics | Module 25 · Notebook 1**

*Prerequisites: Module 01 (NGS Fundamentals), Module 17 (Genome Assembly)*

---

**By the end of this notebook you will be able to:**
1. Explain ONT sequencing chemistry and FAST5/POD5 raw signal format
2. Run basecalling with Dorado and assess read quality with NanoStat/NanoPlot
3. Align long reads to a reference with Minimap2
4. Detect base modifications (5mC methylation) from ONT signal
5. Compare long-read vs short-read alignment statistics

**Key resources:**
- [Oxford Nanopore community tutorials](https://community.nanoporetech.com/)
- [Awesome Nanopore training](https://github.com/GenomicsAotearoa/Genomics-Aotearoa-Nanopore-training)
- [Minimap2 documentation](https://github.com/lh3/minimap2)

---

[← Module Overview](../README.md) | [Next: Assembly & SV →](./02_assembly_sv.ipynb)

---

## 1. ONT Technology Overview

Oxford Nanopore Technology (ONT) sequences DNA by threading a single strand through a protein nanopore embedded in an electrically resistive membrane. A constant voltage drives ionic current through the pore; as each k-mer of bases occupies the constriction zone, it causes a characteristic disruption to that current. The resulting time-series of picoampere values — often called a "squiggle" — encodes the sequence. The R9.4.1 chemistry uses a 5-mer sensing region, while the newer R10.4.1 dual-reader pore uses a 9-mer sensing window, dramatically improving homopolymer resolution and raw-read accuracy.

**FAST5 format** is an HDF5-based container that stores raw signal traces (picoampere arrays), channel metadata, and — if basecalling was run — the basecalled sequence and quality scores. Originally each FAST5 file held data from a single channel/read. Multi-read FAST5 was later introduced but I/O performance remained a bottleneck. FAST5 is now considered legacy.

**POD5 format** is Oxford Nanopore's modern, purpose-built binary format that replaces FAST5. It uses Apache Arrow for columnar on-disk storage, enabling much faster random-access I/O and smaller file sizes. Dorado natively reads POD5. ONT provides `pod5 convert fast5` to migrate legacy data. One POD5 file typically contains reads from an entire batch (not per-channel), simplifying data management.

| Feature | ONT R9.4.1 | ONT R10.4.1 | PacBio HiFi (Revio) |
|---|---|---|---|
| Read length | 50 bp – 4 Mb | 50 bp – 4 Mb | 10–25 kb (CCS) |
| Modal read length | ~8–12 kb | ~10–20 kb | ~15–18 kb |
| Raw accuracy | ~95% | ~97–99% | ~99.9% (CCS) |
| Consensus accuracy | 99.5%+ | 99.9%+ | 99.9%+ |
| Throughput per flow cell | ~30–50 Gb | ~50–120 Gb | ~90 Gb (Revio) |
| Native 5mC detection | Yes (retrained model) | Yes (dual-base calling) | No (requires bisulfite or 6mA) |
| Instrument | MinION / PromethION | MinION R10 / P2 / PromethION | Revio |
| Typical N50 (human WGS) | 15–25 kb | 20–40 kb | 15–20 kb |

## 2. Basecalling with Dorado

**Dorado** is ONT's current GPU-accelerated basecaller, replacing the older Guppy. It uses a convolutional + recurrent neural network trained to translate raw squiggle signals into nucleotide sequences and per-base quality scores. Dorado is open-source and supports both NVIDIA CUDA and Apple Silicon (Metal) backends, making it usable on workstations as well as HPC clusters.

Dorado offers three model accuracy tiers for each chemistry. The **`fast`** model runs at maximum throughput (~95–97% accuracy) and is suitable for rapid screening or real-time adaptive sampling. The **`hac`** (high-accuracy) model provides a good balance of speed and quality (~97–99%) and is the standard choice for most genomic applications. The **`sup`** (super-accuracy) model achieves ~99%+ accuracy but is 5–10× slower than `hac`, making it best suited for post-run reprocessing of critical samples on a GPU cluster.

Output from Dorado is an **unaligned BAM** by default — this is the recommended format because it preserves all per-read metadata (move tables, basecalling model version, sequencing summary statistics) as BAM tags. FASTQ can be obtained by piping through `samtools fastq`. For modification-aware basecalling, append the modification code to the model name (e.g. `hac,5mCG_5hmCG`); probabilities are stored in the MM and ML BAM tags per the SAM specification. **Duplex mode** pairs the template and complement strands from the same DNA molecule, achieving ~Q30 (99.9%) accuracy on the duplex fraction of reads.

```bash
# Quick reference: Dorado model tiers
# fast  → ~95-97% accuracy, highest throughput
# hac   → ~97-99% accuracy, standard choice
# sup   → ~99%+  accuracy, 5-10x slower than hac
```

In [ ]:
# Dorado basecalling — simplex HAC model
# !dorado basecaller hac pod5_data/ > calls.bam
# Explanation: 'hac' = high-accuracy model; output is unaligned BAM

# Convert to FASTQ if needed
# !samtools fastq calls.bam | gzip > calls.fastq.gz

# Duplex basecalling (pairs complementary strands, ~Q30)
# !dorado duplex hac pod5_data/ > calls_duplex.bam

# Methylation-aware: add modification tag to model name
# !dorado basecaller hac,5mCG_5hmCG pod5_data/ > calls_modcall.bam

# Check basecalling summary stats
# !dorado summary calls.bam | head -5

# List available models (requires internet or local model cache)
# !dorado download --list

## 3. Read Quality Assessment

**NanoStat** produces a concise text summary of key run metrics: total reads, total bases sequenced, mean and median read length, N50 read length (the length at which 50% of all sequenced bases are in reads of that length or longer), mean and median Phred quality score, and the percentage of reads passing Q10, Q15, and Q20 thresholds. This is the first QC step after basecalling — a healthy R10.4.1 run typically shows N50 > 15 kb and median quality > Q15.

**NanoPlot** generates a suite of interactive HTML visualizations including a read-length histogram, a quality-score distribution, a yield-over-time plot, and a length-vs-quality scatter ("dot") plot. The dot plot is especially useful for identifying distinct populations (e.g. short adapter-only fragments, ultra-long reads) and for comparing runs. NanoPlot can accept FASTQ, BAM, or a sequencing summary CSV as input.

**NanoFilt** is a streaming FASTQ filter. Typical parameters for whole-genome sequencing assembly are `-q 8 -l 3000` (retain reads ≥ Q8 and ≥ 3 kb). For variant calling where accuracy matters more than depth, `-q 10 -l 1000` removes low-quality and very short reads while retaining the majority of bases. Quality thresholds on the Phred scale: **Q10** = 90% accuracy per base, **Q15** = 96.8%, **Q20** = 99%.

| Quality | Per-base accuracy | Typical ONT chemistry |
|---|---|---|
| Q10 | 90.0% | R9.4.1 minimum |
| Q12 | 93.7% | R9.4.1 median |
| Q15 | 96.8% | R10.4.1 median |
| Q20 | 99.0% | R10.4.1 sup / duplex |
| Q30 | 99.9% | ONT duplex / PacBio CCS |

In [ ]:
# Run NanoStat on raw FASTQ
# !NanoStat --fastq calls.fastq.gz --outdir nanostat_out/ --threads 4

# NanoPlot (produces interactive HTML)
# !NanoPlot --fastq calls.fastq.gz --outdir nanoplot_out/ --plots dot --N50

# Filter reads: min quality Q10, min length 1000 bp
# !NanoFilt -q 10 -l 1000 calls.fastq.gz | gzip > calls_filtered.fastq.gz

# Check number of reads before/after filtering
# !echo "Before: $(zcat calls.fastq.gz | wc -l | awk '{print $1/4}') reads"
# !echo "After:  $(zcat calls_filtered.fastq.gz | wc -l | awk '{print $1/4}') reads"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Simulate ONT R10.4.1 read length distribution (log-normal) and quality
n_reads = 5000
# Read lengths: log-normal with mean ~15 kb
lengths = rng.lognormal(mean=9.6, sigma=0.9, size=n_reads).astype(int)
lengths = np.clip(lengths, 200, 500_000)

# Quality scores: approximately normal around Q16 for R10.4.1
mean_q = rng.normal(16.5, 2.5, size=n_reads)
mean_q = np.clip(mean_q, 5, 30)

df_reads = pd.DataFrame({"length_bp": lengths, "mean_q": mean_q})

# --- compute NanoStat-style summary ---
sorted_len = np.sort(lengths)[::-1]
cumsum = np.cumsum(sorted_len)
n50_val = sorted_len[np.searchsorted(cumsum, cumsum[-1] / 2)]

print("=== Simulated NanoStat Summary ===")
print(f"Total reads:          {n_reads:>10,}")
print(f"Total bases (Gb):     {lengths.sum()/1e9:>10.2f}")
print(f"Mean read length:     {lengths.mean():>10,.0f} bp")
print(f"Median read length:   {np.median(lengths):>10,.0f} bp")
print(f"N50 read length:      {n50_val:>10,} bp")
print(f"Mean read quality:    {mean_q.mean():>10.1f}")
print(f"Reads > Q10:          {(mean_q >= 10).mean()*100:>9.1f}%")
print(f"Reads > Q15:          {(mean_q >= 15).mean()*100:>9.1f}%")
print(f"Reads > Q20:          {(mean_q >= 20).mean()*100:>9.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Panel 1: read length histogram (log scale)
axes[0].hist(lengths / 1000, bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
axes[0].axvline(n50_val / 1000, color="crimson", linestyle="--", label=f"N50 = {n50_val/1000:.1f} kb")
axes[0].set_xlabel("Read length (kb)")
axes[0].set_ylabel("Count")
axes[0].set_title("Read length distribution")
axes[0].set_xscale("log")
axes[0].legend()

# Panel 2: quality distribution
axes[1].hist(mean_q, bins=40, color="darkorange", edgecolor="white", linewidth=0.3)
axes[1].axvline(10, color="red", linestyle="--", label="Q10 threshold")
axes[1].axvline(15, color="green", linestyle="--", label="Q15 threshold")
axes[1].set_xlabel("Mean read quality (Phred)")
axes[1].set_ylabel("Count")
axes[1].set_title("Quality score distribution")
axes[1].legend()

# Panel 3: length vs quality scatter
sc = axes[2].scatter(lengths / 1000, mean_q, alpha=0.05, s=4, c=mean_q, cmap="RdYlGn")
axes[2].set_xlabel("Read length (kb)")
axes[2].set_ylabel("Mean quality")
axes[2].set_title("Length vs. quality")
axes[2].set_xscale("log")
plt.colorbar(sc, ax=axes[2], label="Q score")

plt.suptitle("ONT R10.4.1 — simulated read QC (5,000 reads)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Alignment with Minimap2

**Minimap2** is the standard aligner for long reads. Rather than aligning every position like short-read aligners (BWA, Bowtie2), Minimap2 uses **minimizer-based seeding**: it extracts a sparse set of representative k-mers (minimizers) from both query and reference, finds collinear chains of matching minimizers, then performs banded dynamic programming only in the seed-chain regions. This makes alignment of 10–100 kb reads orders of magnitude faster than classical approaches.

**Alignment presets** (`-ax` flag) configure Minimap2's parameters for different data types. `map-ont` is optimised for Oxford Nanopore genomic reads (higher mismatch tolerance, no splice-aware scoring). `map-hifi` tunes for PacBio HiFi's higher accuracy and read length distribution. `splice` enables intron-aware alignment for RNA/cDNA reads, recognising the N CIGAR operation for intron skipping. `ava-ont` is used for all-vs-all overlap computation during de novo assembly. The preset can make a 5–10× difference in sensitivity and specificity.

Long reads typically achieve **95–99% alignment rate** for genomic DNA sequencing when the sample is the same species as the reference. The supplementary alignment rate (reads split across two loci — a signature of structural variants) is ~1–5%, compared with <0.1% for short reads. After alignment, always sort with `samtools sort` and index with `samtools index` for efficient random access. Use `samtools flagstat` for a quick summary of mapping statistics.

```
Key samtools flagstat fields for long-read BAMs:
  - total reads            → all primary + supplementary + secondary
  - mapped (%)             → primary mapped reads
  - supplementary          → chimeric/split reads (SV signatures)
  - paired-in-sequencing   → always 0 for long reads (single-end)
```

In [ ]:
# Align ONT reads (map-ont preset)
# !minimap2 -ax map-ont -t 8 hg38.fa calls_filtered.fastq.gz \
#     | samtools sort -o ont_aligned.bam -@ 8
# !samtools index ont_aligned.bam

# Check alignment statistics
# !samtools flagstat ont_aligned.bam

# For PacBio HiFi reads use map-hifi preset:
# !minimap2 -ax map-hifi -t 8 hg38.fa hifi_reads.fastq.gz \
#     | samtools sort -o hifi_aligned.bam

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

# Simulate alignment statistics for two platforms
platforms = {
    "ONT R10.4.1": {
        "total_reads": 200_000,
        "mapped_pct": 97.3,
        "mean_length_kb": 16.2,
        "mean_identity": 98.5,
        "supplementary_pct": 3.1,
    },
    "Illumina 150 bp PE": {
        "total_reads": 40_000_000,
        "mapped_pct": 99.2,
        "mean_length_kb": 0.15,
        "mean_identity": 99.8,
        "supplementary_pct": 0.1,
    },
}

df_plat = pd.DataFrame(platforms).T
print("=== Alignment comparison: ONT vs Illumina ===")
print(df_plat[["total_reads","mapped_pct","mean_length_kb","mean_identity","supplementary_pct"]].to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ONT length distribution
ont_lengths = rng.lognormal(9.6, 0.85, 5000)
ont_lengths = np.clip(ont_lengths, 200, 1_000_000)
axes[0].hist(ont_lengths / 1000, bins=60, color="steelblue", edgecolor="white", lw=0.3)
axes[0].set_xlabel("Aligned read length (kb)")
axes[0].set_ylabel("Count")
axes[0].set_title("ONT R10.4.1 aligned read lengths")
axes[0].set_xscale("log")
sorted_ont = np.sort(ont_lengths)[::-1]
n50_ont = sorted_ont[np.searchsorted(np.cumsum(sorted_ont), np.cumsum(sorted_ont)[-1] / 2)]
axes[0].axvline(n50_ont / 1000, color="crimson", linestyle="--", label=f"N50 \u2248 {n50_ont/1000:.1f} kb")
axes[0].legend()

# Identity distribution (per-read)
identity = rng.beta(49, 1, 5000) * 100   # peak ~98%
axes[1].hist(identity, bins=40, color="seagreen", edgecolor="white", lw=0.3)
axes[1].set_xlabel("Per-read alignment identity (%)")
axes[1].set_ylabel("Count")
axes[1].set_title("ONT R10.4.1 alignment identity")
axes[1].axvline(np.median(identity), color="crimson", linestyle="--",
                label=f"Median = {np.median(identity):.1f}%")
axes[1].legend()

plt.suptitle("Minimap2 alignment statistics (simulated)", fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Base Modification Detection

One of ONT's most powerful and unique capabilities is **direct detection of base modifications** from the raw electrical signal — without requiring bisulfite conversion, chemical enrichment, or pull-down. As a methylated cytosine (5mC) or hydroxymethylcytosine (5hmC) passes through the pore, it produces a slightly different current signature than its unmodified counterpart. Dorado's modification-aware models are trained to distinguish these signatures.

The workflow has two stages. First, **Dorado basecalls with a modification model** by appending the modification code to the model name: `dorado basecaller hac,5mCG_5hmCG pod5/`. This produces an unaligned BAM where each read carries `MM` (modification metadata) and `ML` (modification likelihood per position) tags as defined in the SAM specification. These tags are preserved through alignment. Second, **`modkit pileup`** aggregates per-read modification probabilities across all reads aligned to each reference position, producing a **bedMethyl** file with per-CpG methylation frequency.

The **bedMethyl format** has 11 columns: `chrom`, `start`, `end`, `name`, `score`, `strand`, `thickStart`, `thickEnd`, `itemRgb`, `coverage`, `percentModified`. Use `--cpg` to restrict output to CpG context and `--combine-strands` to merge forward/reverse strand calls into a single symmetric CpG value. Typical methylation patterns: active gene promoters at CpG islands are **0–20% methylated** (hypomethylated), gene bodies are **50–80%**, and intergenic/repeat regions are **80–100%** (hypermethylated).

Key applications of ONT methylation data include: detecting CpG island methylation state, identifying imprinted genes, cancer methylation signatures, chromatin accessibility footprinting (CTCF binding sites appear as methylation-depleted footprints), and allele-specific methylation phasing.

```
bedMethyl column reference:
  col 1-3:  chrom, start, end (0-based half-open)
  col 10:   coverage (number of reads with valid call)
  col 11:   percent_modified (0–100 scale)
```

In [ ]:
# Step 1: basecall with modification model
# !dorado basecaller hac,5mCG_5hmCG pod5_data/ > calls_mod.bam

# Step 2: align the modcall BAM (MM/ML tags are preserved through alignment)
# !minimap2 -ax map-ont -t 8 --MD hg38.fa calls_mod.bam \
#     | samtools sort -o mod_aligned.bam
# !samtools index mod_aligned.bam

# Step 3: pileup CpG methylation frequencies with modkit
# !modkit pileup mod_aligned.bam methylation.bed \
#     --ref hg38.fa --cpg --combine-strands --threads 8
# Output: bedMethyl format  chrom  start  end  name  score  strand  ...  coverage  freq

# Preview the output
# !head -5 methylation.bed

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(7)

# Simulate a 50 kb region with mixed methylation patterns
region_start = 1_000_000
n_cpg = 400
cpg_positions = np.sort(rng.choice(np.arange(region_start, region_start + 50_000), n_cpg, replace=False))

# Assign methylation context zones
# Zone 0: CpG island (active promoter) — hypomethylated
# Zone 1: gene body — moderately methylated
# Zone 2: intergenic — hypermethylated
zones = np.digitize(cpg_positions, [region_start + 10_000, region_start + 35_000])
base_methyl = np.array([0.08, 0.62, 0.88])[zones]
noise = rng.normal(0, 0.06, n_cpg)
methyl_freq = np.clip(base_methyl + noise, 0, 1)
coverage = rng.poisson(35, n_cpg)

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

# Methylation frequency track
sc = axes[0].scatter(
    (cpg_positions - region_start) / 1000,
    methyl_freq * 100,
    c=methyl_freq * 100,
    cmap="RdYlGn_r",
    s=12, alpha=0.7
)
plt.colorbar(sc, ax=axes[0], label="5mC frequency (%)")
axes[0].set_ylabel("CpG methylation (%)")
axes[0].set_title("modkit pileup output — CpG methylation frequency (simulated)")
axes[0].axhline(50, color="gray", linestyle=":", alpha=0.5)
for label, start, end, color in [
    ("CpG island\n(active promoter)", 0, 10, "lightblue"),
    ("Gene body", 10, 35, "lightyellow"),
    ("Intergenic", 35, 50, "lightsalmon"),
]:
    axes[0].axvspan(start, end, alpha=0.15, color=color, label=label)
axes[0].legend(fontsize=8, loc="upper right")

# Coverage track
axes[1].bar(
    (cpg_positions - region_start) / 1000,
    coverage, width=0.08, color="steelblue", alpha=0.7
)
axes[1].set_xlabel("Genomic position (kb from region start)")
axes[1].set_ylabel("Coverage (reads)")
axes[1].set_title("Read depth at CpG sites")

plt.tight_layout()
plt.show()

print("\nMedian methylation by zone:")
print(f"  CpG island (0-10 kb):  {methyl_freq[zones==0].mean()*100:.1f}%  (n={( zones==0).sum()} CpGs)")
print(f"  Gene body (10-35 kb):  {methyl_freq[zones==1].mean()*100:.1f}%  (n={(zones==1).sum()} CpGs)")
print(f"  Intergenic (35-50 kb): {methyl_freq[zones==2].mean()*100:.1f}%  (n={(zones==2).sum()} CpGs)")

## Summary

This notebook walked through the full ONT data processing pipeline: from raw electrical signal stored in POD5 files, through GPU-accelerated basecalling with Dorado (simplex and duplex modes, with optional methylation calling), quality control with NanoStat/NanoPlot, read filtering with NanoFilt, reference alignment with Minimap2, and finally per-position CpG methylation pileup with modkit.

**Long-read sequencing advantages over short reads:**
- **Structural variant detection**: reads spanning SVs (deletions, inversions, duplications, translocations) directly; typical sensitivity 3–5× higher than short-read SV callers
- **Native methylation**: no bisulfite conversion; simultaneous sequencing + epigenome profiling from a single library
- **Phasing**: reads long enough to span multiple heterozygous SNPs → haplotype-resolved assemblies and allele-specific methylation
- **Complex/repetitive regions**: centromeres, telomeres, segmental duplications, satellite repeats — inaccessible to short reads
- **Full-length isoform sequencing**: no assembly of fragmented reads required (see Module 25 Notebook 3)

The next notebook covers **de novo genome assembly and structural variant calling** using long reads.

| Step | Tool | Key Command | Output | Notes |
|---|---|---|---|---|
| Signal acquisition | MinKNOW | (device software) | POD5 files | Raw ionic current per read |
| Basecalling | Dorado `hac` | `dorado basecaller hac pod5/ > calls.bam` | Unaligned BAM | Add `,5mCG_5hmCG` for methylation |
| Quality control | NanoStat + NanoPlot | `NanoStat --fastq calls.fastq.gz` | Text summary + HTML | Check N50, mean Q, length dist. |
| Read filtering | NanoFilt | `NanoFilt -q 10 -l 1000` | Filtered FASTQ | Remove short/low-quality reads |
| Alignment (DNA) | Minimap2 | `minimap2 -ax map-ont hg38.fa` | Sorted BAM | Use `map-hifi` for PacBio HiFi |
| Alignment (RNA) | Minimap2 | `minimap2 -ax splice hg38.fa` | Sorted BAM | Recognises intron CIGAR N ops |
| Methylation pileup | modkit | `modkit pileup --cpg` | bedMethyl BED | Requires MM/ML tags from Dorado |

In [ ]:
# ── Exercise 25-1: ONT Data Processing ─────────────────────────────────────
#
# 1. SIGNAL FORMAT: A colleague gives you FAST5 files from an older run.
#    What tool can convert FAST5 to POD5? (Hint: ONT provides pod5-tools)
#    Why would you want to convert before re-basecalling with Dorado?
#
# 2. MODEL CHOICE: You are sequencing a clinical sample and need maximum
#    accuracy for SNP calling. Which Dorado model should you use?
#    What is the approximate trade-off in sequencing throughput?
#
# 3. QC THRESHOLDS (run the simulation below):

import numpy as np
rng = np.random.default_rng(99)
n = 10_000
lengths = rng.lognormal(9.2, 1.1, n).astype(int)
quals   = rng.normal(14.5, 3.0, n)
quals   = np.clip(quals, 3, 30)

# Q3a: What fraction of reads pass NanoFilt -q 10 -l 1000?
pass_filter = (quals >= 10) & (lengths >= 1000)
print(f"Reads passing -q10 -l1000: {pass_filter.mean()*100:.1f}%")

# Q3b: Compute the N50 of the PASSING reads only
passing_lengths = np.sort(lengths[pass_filter])[::-1]
cumsum = np.cumsum(passing_lengths)
n50 = passing_lengths[np.searchsorted(cumsum, cumsum[-1] / 2)]
print(f"N50 of passing reads: {n50:,} bp = {n50/1000:.1f} kb")

# Q3c: What percentage of total bases are retained?
total_bases_before = lengths.sum()
total_bases_after  = lengths[pass_filter].sum()
print(f"Bases retained: {total_bases_after/total_bases_before*100:.1f}%")

# 4. METHYLATION: bedMethyl format has columns:
#    chrom, start, end, name, score, strand, thickStart, thickEnd, rgb, coverage, pct_modified
#    Write a pandas one-liner to load methylation.bed and filter for:
#    (a) coverage >= 10 reads  (b) CpGs on chr1 only
#    How would you compute the mean methylation of CpGs overlapping a promoter BED file?

---

[← Module Overview](../README.md) | [Next: Assembly & SV →](./02_assembly_sv.ipynb)

---